In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_data(file_path):
    """Load UCI SMS SpamCollection (tab-separated: label\ttext)."""
    # Read tab-separated, names for clarity
    df = pd.read_csv(file_path, sep='\t', header=None, names=['label_str', 'text'])
    df['label'] = LabelEncoder().fit_transform(df['label_str'])  # ham=0, spam=1
    df = df[['label', 'text']]  # Drop string label
    print(f"Loaded {len(df)} samples: {df['label'].value_counts().to_dict()}")
    return df

def preprocess_data(df):
    """Lowercase text (TF-IDF handles rest)."""
    df['text'] = df['text'].str.lower().str.strip()
    # Remove any NaN/empty (rare)
    df = df.dropna(subset=['text']).query('text != ""')
    return df[['label', 'text']]

def split_and_store(df, test_size=0.2, val_size=0.2, random_state=42):
    """64/16/20 stratified split."""
    train_val_test = train_test_split(df, test_size=test_size, 
                                      stratify=df['label'], random_state=random_state)
    train_val, test = train_val_test[0], train_val_test[1]
    train, val = train_test_split(train_val, test_size=val_size/(1-test_size), 
                                  stratify=train_val['label'], random_state=random_state)
    train.to_csv('train.csv', index=False)
    val.to_csv('validation.csv', index=False)
    test.to_csv('test.csv', index=False)
    print(f"Splits: train={len(train)}, val={len(val)}, test={len(test)}")
    print("Train dist:", train['label'].value_counts(normalize=True))


In [3]:
df = load_data(r"C:\Users\Ahana\Downloads\sms+spam+collection\SMSSpamCollection")
df = preprocess_data(df)
split_and_store(df)

Loaded 5572 samples: {0: 4825, 1: 747}
Splits: train=3342, val=1115, test=1115
Train dist: label
0    0.865949
1    0.134051
Name: proportion, dtype: float64


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Benchmark models (simple, effective for text)
models = {
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000),
    'NaiveBayes': MultinomialNB(),
    'SVC': SVC(random_state=42, probability=True)
}

def fit_model(model, train_path):
    """Fit pipeline on train CSV."""
    df_train = pd.read_csv(train_path)
    pipe = Pipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), 
                     ('clf', model)])
    pipe.fit(df_train['text'], df_train['label'])
    return pipe

def score_model(model_pipe, data_path):
    """Predict probabilities/labels, return accuracy/F1."""
    df = pd.read_csv(data_path)
    preds = model_pipe.predict(df['text'])
    probs = model_pipe.predict_proba(df['text'])[:, 1]
    return {'accuracy': accuracy_score(df['label'], preds),
            'f1': f1_score(df['label'], preds), 'probs': probs}

def evaluate_predictions(y_true, preds, probs=None):
    """Print classification report."""
    print(classification_report(y_true, preds))
    if probs is not None:
        print(f"Test Accuracy: {accuracy_score(y_true, preds):.3f}")

def validate_model(model_name):
    """Train on train.csv, score/eval on train+val."""
    model = models[model_name]
    pipe = fit_model(model, 'train.csv')
    
    # Train scores
    train_scores = score_model(pipe, 'train.csv')
    print(f"{model_name} Train: Acc {train_scores['accuracy']:.3f}, F1 {train_scores['f1']:.3f}")
    evaluate_predictions(pd.read_csv('train.csv')['label'], 
                        pipe.predict(pd.read_csv('train.csv')['text']))
    
    # Val scores
    val_scores = score_model(pipe, 'validation.csv')
    print(f"{model_name} Val: Acc {val_scores['accuracy']:.3f}, F1 {val_scores['f1']:.3f}")
    evaluate_predictions(pd.read_csv('validation.csv')['label'], 
                        pipe.predict(pd.read_csv('validation.csv')['text']))
    
    return pipe, train_scores, val_scores

def fine_tune_hyperparams(model_name):
    """Optional GridSearchCV on train+val for LogisticRegression (best baseline often)."""
    from sklearn.model_selection import GridSearchCV
    df_train = pd.read_csv('train.csv')
    df_val = pd.read_csv('validation.csv')
    X_trainval = pd.concat([df_train['text'], df_val['text']])
    y_trainval = pd.concat([df_train['label'], df_val['label']])
    
    if model_name == 'LogisticRegression':
        param_grid = {'clf__C': [0.1, 1, 10], 'tfidf__max_features': [3000, 5000]}
    else:
        return None  # Skip for others (simple hyperparameters)
    
    pipe = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2))), ('clf', models[model_name])])
    grid = GridSearchCV(pipe, param_grid, cv=3, scoring='f1')
    grid.fit(X_trainval, y_trainval)
    print(f"Best params for {model_name}: {grid.best_params_}")
    return grid.best_estimator_

def benchmark_test():
    """Score all 3 models on test.csv, select best by F1."""
    best_f1, best_model, best_pipe = 0, None, None
    test_df = pd.read_csv('test.csv')
    
    for name in models:
        pipe = fit_model(models[name], 'train.csv')  # Retrain on train only
        scores = score_model(pipe, 'test.csv')
        print(f"{name} Test: Acc {scores['accuracy']:.3f}, F1 {scores['f1']:.3f}")
        evaluate_predictions(test_df['label'], pipe.predict(test_df['text']), scores['probs'])
        
        if scores['f1'] > best_f1:
            best_f1, best_model, best_pipe = scores['f1'], name, pipe
    
    print(f"Best model: {best_model} (F1: {best_f1:.3f})")
    return best_pipe

In [5]:
# Validate 3 models
for name in ['LogisticRegression', 'NaiveBayes', 'SVC']:
    validate_model(name)

# Optional: Tune top val performer (e.g., SVC)
# tuned = fine_tune_hyperparams('SVC')

# Benchmark on test (uses train fit)
best_pipe = benchmark_test()


LogisticRegression Train: Acc 0.977, F1 0.907
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      2894
           1       1.00      0.83      0.91       448

    accuracy                           0.98      3342
   macro avg       0.99      0.92      0.95      3342
weighted avg       0.98      0.98      0.98      3342

LogisticRegression Val: Acc 0.966, F1 0.855
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       965
           1       1.00      0.75      0.85       150

    accuracy                           0.97      1115
   macro avg       0.98      0.87      0.92      1115
weighted avg       0.97      0.97      0.96      1115

NaiveBayes Train: Acc 0.983, F1 0.931
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      2894
           1       1.00      0.87      0.93       448

    accuracy                           0.98      3342
